In [4]:
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.employee.dq")

# ── Read Bronze ───────────────────────────────────────────────────────
df_employee = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_Employee")
df_person = spark.read.table("lh_adventureworks_bronze.dbo.Person_Person")
df_dept = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_Department")

# Note: EmployeeDepartmentHistory wasn't in original Bronze list —
# checking if it exists; if not, we'll handle department differently
try:
    df_edh = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_EmployeeDepartmentHistory")
    has_edh = True
except Exception:
    has_edh = False
    print("Note: EmployeeDepartmentHistory not in Bronze — department link will be skipped for now")

logger.info(f"Employee rows  : {df_employee.count()}")
logger.info(f"Person rows  : {df_person.count()}")
logger.info(f"Department rows  : {df_dept.count()}")
logger.info(f"Has EDH table  : {has_edh}")

StatementMeta(, 2df620d0-0ba1-4ca6-80ef-32a4bd53123c, 6, Finished, Available, Finished, False)

INFO:silver.employee.dq:Employee rows  : 290
INFO:silver.employee.dq:Person rows  : 19972
INFO:silver.employee.dq:Department rows  : 16
INFO:silver.employee.dq:Has EDH table  : True


In [7]:
from pyspark.sql import functions as F

# ── Join Employee + Person (name resolution) ──────────────────────────
df_person_clean = df_person.select(
    F.col("BusinessEntityID").alias("p_BusinessEntityID"),
    F.col("FirstName"),
    F.col("MiddleName"),
    F.col("LastName"),
    F.col("EmailPromotion")
)

df_silver_employee = df_employee.join(
    df_person_clean,
    df_employee["BusinessEntityID"] == df_person_clean["p_BusinessEntityID"],
    how="left"
).drop("p_BusinessEntityID")

# ── Join EmployeeDepartmentHistory — keep only CURRENT assignment ────
# An employee can have multiple rows in EDH over time; EndDate IS NULL = current
df_edh_current = df_edh.filter(F.col("EndDate").isNull()).select(
    F.col("BusinessEntityID").alias("edh_BusinessEntityID"),
    F.col("DepartmentID"),
    F.col("ShiftID"),
    F.col("StartDate").alias("DepartmentStartDate")
)

df_silver_employee = df_silver_employee.join(
    df_edh_current,
    df_silver_employee["BusinessEntityID"] == df_edh_current["edh_BusinessEntityID"],
    how="left"
).drop("edh_BusinessEntityID")

# ── Join Department (name resolution) ─────────────────────────────────
df_dept_clean = df_dept.select(
    F.col("DepartmentID").alias("d_DepartmentID"),
    F.col("Name").alias("DepartmentName"),
    F.col("GroupName")
)

df_silver_employee = df_silver_employee.join(
    df_dept_clean,
    df_silver_employee["DepartmentID"] == df_dept_clean["d_DepartmentID"],
    how="left"
).drop("d_DepartmentID")

# ── Build full name + select final columns ────────────────────────────
df_silver_employee = df_silver_employee.withColumn(
    "EmployeeFullName",
    F.concat_ws(" ", F.col("FirstName"), F.col("MiddleName"), F.col("LastName"))
)

df_silver_employee = df_silver_employee.select(
    F.col("BusinessEntityID").alias("EmployeeID"),
    "EmployeeFullName",
    "FirstName",
    "LastName",
    F.col("NationalIDNumber"),
    F.col("LoginID"),
    F.col("OrganizationNode"),
    F.col("OrganizationLevel"),
    F.col("JobTitle"),
    F.col("BirthDate"),
    F.col("MaritalStatus"),
    F.col("Gender"),
    F.col("HireDate"),
    F.col("SalariedFlag"),
    F.col("VacationHours"),
    F.col("SickLeaveHours"),
    F.col("CurrentFlag"),
    "DepartmentID",
    "DepartmentName",
    "GroupName",
    "ShiftID",
    "DepartmentStartDate",
    F.col("rowguid"),
    F.col("ModifiedDate")
)
# ── Type standardization (catch the remaining shorts) ──────────────────
df_silver_employee = df_silver_employee \
    .withColumn("DepartmentID", F.col("DepartmentID").cast("integer")) \
    .withColumn("ShiftID", F.col("ShiftID").cast("integer")) \
    .withColumn("OrganizationLevel", F.col("OrganizationLevel").cast("integer"))


StatementMeta(, 2df620d0-0ba1-4ca6-80ef-32a4bd53123c, 9, Finished, Available, Finished, False)

In [8]:
# ── DQ checks ─────────────────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.employee.dq")

dq_passed = True
dq_results = []

def log_check(name, passed, message):
    global dq_passed
    status = "PASS" if passed else "FAIL"
    log_msg = f"[DQ] [{status}] {name} — {message}"
    if passed:
        logger.info(log_msg)
    else:
        logger.error(log_msg)
        dq_passed = False
    dq_results.append({"check": name, "status": status, "message": message})

actual_rows = df_silver_employee.count()

log_check(
    "Row count",
    actual_rows == df_employee.count(),
    f"Expected {df_employee.count():,}, got {actual_rows:,}"
)

log_check(
    "Null PK — EmployeeID",
    df_silver_employee.filter(F.col("EmployeeID").isNull()).count() == 0,
    f"{df_silver_employee.filter(F.col('EmployeeID').isNull()).count():,} nulls"
)

dup_pk = actual_rows - df_silver_employee.select("EmployeeID").distinct().count()
log_check("Duplicate PK — EmployeeID", dup_pk == 0, f"{dup_pk:,} duplicates")

unmatched_dept = df_silver_employee.filter(
    F.col("DepartmentID").isNotNull() & F.col("DepartmentName").isNull()
).count()
log_check(
    "Department join coverage",
    unmatched_dept == 0,
    f"{unmatched_dept:,} employees with DepartmentID but no DepartmentName"
)

no_dept_at_all = df_silver_employee.filter(F.col("DepartmentID").isNull()).count()
log_check(
    "Employees with no current department (informational)",
    True,  # informational only, doesn't fail the pipeline
    f"{no_dept_at_all:,} employees have no current EDH row (EndDate IS NULL)"
)

failed_checks = [r for r in dq_results if r["status"] == "FAIL"]
if failed_checks:
    failed_names = ", ".join([r["check"] for r in failed_checks])
    raise Exception(
        f"silver.employee DQ failed — {len(failed_checks)} check(s) failed: "
        f"{failed_names}. Pipeline halted."
    )

logger.info(f"[DQ] All checks passed — {actual_rows:,} rows cleared for Silver write.")

StatementMeta(, 2df620d0-0ba1-4ca6-80ef-32a4bd53123c, 10, Finished, Available, Finished, False)

INFO:silver.employee.dq:[DQ] [PASS] Row count — Expected 290, got 290
INFO:silver.employee.dq:[DQ] [PASS] Null PK — EmployeeID — 0 nulls
INFO:silver.employee.dq:[DQ] [PASS] Duplicate PK — EmployeeID — 0 duplicates
INFO:silver.employee.dq:[DQ] [PASS] Department join coverage — 0 employees with DepartmentID but no DepartmentName
INFO:silver.employee.dq:[DQ] [PASS] Employees with no current department (informational) — 0 employees have no current EDH row (EndDate IS NULL)
INFO:silver.employee.dq:[DQ] All checks passed — 290 rows cleared for Silver write.


In [9]:
# ── Write ──────────────────────────────────────────────────────────────
#spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.hr")

df_silver_employee.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.hr.employee")

df_verify = spark.read.table("lh_adventureworks_silver.hr.employee")
#print(f"silver.hr.employee written — {df_verify.count():,} rows verified.")

StatementMeta(, 2df620d0-0ba1-4ca6-80ef-32a4bd53123c, 11, Finished, Available, Finished, False)

silver.hr.employee written — 290 rows verified.
